# Day 50 — Evaluation metrics and cross-validation

A classifier hands you a score, not a decision. Everything you then report about it —
accuracy, precision, recall, F1, AUC — is a different ratio taken from the same four
counts, and choosing one of them is choosing which mistakes you are willing to make.

This notebook measures that choice. A model that has clearly learned something scores
worse than a constant. Two models are ranked in opposite orders by the two curves
everyone quotes. A fold ends up with no positives in it. And a pipeline that looks
careful reports 0.9900 for a model that is worth 0.4750.

In [1]:
import math

import numpy as np


# ====================================================== 1. the four counts
def confusion(y_true, y_pred):
    """Return (tn, fp, fn, tp) for 0/1 labels.

    Everything else on this page is one ratio of these four numbers.  They are
    counts, so they never go undefined; the ratios built from them can.
    """
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    return tn, fp, fn, tp


def accuracy(y_true, y_pred):
    tn, fp, fn, tp = confusion(y_true, y_pred)
    return (tp + tn) / (tp + tn + fp + fn)


def precision(y_true, y_pred):
    """Of the rows we flagged, how many were right?  Undefined when we flag nothing."""
    tn, fp, fn, tp = confusion(y_true, y_pred)
    return float('nan') if tp + fp == 0 else tp / (tp + fp)


def recall(y_true, y_pred):
    """Of the rows that really are positive, how many did we catch?"""
    tn, fp, fn, tp = confusion(y_true, y_pred)
    return float('nan') if tp + fn == 0 else tp / (tp + fn)


def f1(y_true, y_pred):
    """Harmonic mean of precision and recall - 0 when either one is 0."""
    p, r = precision(y_true, y_pred), recall(y_true, y_pred)
    if math.isnan(p) or math.isnan(r) or p + r == 0:
        return 0.0 if not (math.isnan(p) or math.isnan(r)) else float('nan')
    return 2 * p * r / (p + r)


def specificity(y_true, y_pred):
    """Of the rows that really are negative, how many did we leave alone?"""
    tn, fp, fn, tp = confusion(y_true, y_pred)
    return float('nan') if tn + fp == 0 else tn / (tn + fp)


def balanced_accuracy(y_true, y_pred):
    """Mean of recall and specificity: each class gets one vote regardless of size."""
    return (recall(y_true, y_pred) + specificity(y_true, y_pred)) / 2


def mcc(y_true, y_pred):
    """Matthews correlation: uses all four counts, so it cannot be fooled by
    predicting one class everywhere (that makes the denominator 0 -> nan here)."""
    tn, fp, fn, tp = confusion(y_true, y_pred)
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return float('nan') if den == 0 else (tp * tn - fp * fn) / den


def predict_at(scores, threshold):
    """A score becomes a decision only when you pick a threshold.  0.5 is a pick."""
    return (np.asarray(scores) >= threshold).astype(int)


y = np.array([1, 1, 1, 0, 0, 0, 0, 0])
p = np.array([1, 1, 0, 1, 0, 0, 0, 0])
tn, fp, fn, tp = confusion(y, p)
print(f'tn {tn}   fp {fp}   fn {fn}   tp {tp}')
for nm, v in (('accuracy', accuracy(y, p)), ('precision', precision(y, p)),
              ('recall', recall(y, p)), ('specificity', specificity(y, p)),
              ('F1', f1(y, p)), ('balanced accuracy', balanced_accuracy(y, p)),
              ('MCC', mcc(y, p))):
    print(f'{nm:<18} {v:.4f}')

tn 4   fp 1   fn 1   tp 2
accuracy           0.7500
precision          0.6667
recall             0.6667
specificity        0.8000
F1                 0.6667
balanced accuracy  0.7333
MCC                0.4667


## accuracy, precision, recall — and an apple that makes the difference stick

Four ratios, four different denominators. 100 pieces of fruit, 10 of them apples,
and a detector that puts 8 pieces on the apple side — 6 apples and 2 oranges.

- **accuracy** divides by *everything*
- **preci-si-on** divides by the **si**de you picked — what the model said
- **rec-all** divides by **all** the apples — what the truth said
- **FPR** is the mirror on the other class: of all the oranges, how many did you grab

Recall counts apples only and FPR counts oranges only. Precision counts the pieces *you*
put on the apple side, so the model decides how big its denominator is. Accuracy counts
the whole basket, which the dataset decides and the model cannot change — with 90 oranges
in 100 pieces of fruit, most of accuracy is settled before the detector is switched on.

In [2]:
def false_positive_rate(y_true, y_pred):
    """fp / (fp + tn): of the rows that really are negative, how many did we flag?

    This is the x-axis of the ROC curve.  Like recall it is computed entirely inside
    one class, which is why both ROC axes barely notice the class balance.
    """
    tn, fp, fn, tp = confusion(y_true, y_pred)
    return float('nan') if tn + fp == 0 else fp / (tn + fp)


def apples_and_oranges(n=100, n_apples=10, caught=6, wrong=2):
    """A basket of fruit and an apple detector, for the four denominators.

    100 pieces of fruit, 10 of them apples.  The detector puts `caught` apples and
    `wrong` oranges on the apple side.  Returns (truth, prediction).
    """
    y = np.zeros(n, dtype=int)
    y[:n_apples] = 1
    p = np.zeros(n, dtype=int)
    p[:caught] = 1                              # apples it got right
    p[n_apples:n_apples + wrong] = 1            # oranges it called apples
    return y, p


ya, pa = apples_and_oranges()
tn, fp, fn, tp = confusion(ya, pa)
print(f'tp {tp}   fp {fp}   fn {fn}   tn {tn}')
print(f'accuracy  = (tp + tn) / 100  = {accuracy(ya, pa):.4f}   of EVERYTHING')
print(f'precision = tp / (tp + fp)   = {precision(ya, pa):.4f}   of the APPLE SIDE')
print(f'recall    = tp / (tp + fn)   = {recall(ya, pa):.4f}   of ALL THE APPLES')
print(f'FPR       = fp / (fp + tn)   = {false_positive_rate(ya, pa):.4f}   of ALL THE ORANGES')
print(f'F1                           = {f1(ya, pa):.4f}')
print()
none = np.zeros(len(ya), dtype=int)
print('call every piece of fruit an orange:')
print(f'  accuracy {accuracy(ya, none):.4f}   recall {recall(ya, none):.4f}   '
      f'precision {precision(ya, none)}')

tp 6   fp 2   fn 4   tn 88
accuracy  = (tp + tn) / 100  = 0.9400   of EVERYTHING
precision = tp / (tp + fp)   = 0.7500   of the APPLE SIDE
recall    = tp / (tp + fn)   = 0.6000   of ALL THE APPLES
FPR       = fp / (fp + tn)   = 0.0222   of ALL THE ORANGES
F1                           = 0.6667

call every piece of fruit an orange:
  accuracy 0.9000   recall 0.0000   precision nan


## A model that learned something, scoring worse than doing nothing

2,000 rows, 20 of them positive. Logistic regression, scored on the rows it was not
trained on. At the default cut of 0.5 it is beaten by a constant — and the same scores
give a ROC-AUC that says it ranks the positives well. Both readings are arithmetically
correct; the cut is what nobody chose.

In [3]:
# ====================================================== 5. datasets
def imbalanced(n=2000, pos_rate=0.01, sep=1.6, d=6, seed=0):
    """n rows, a given fraction of them positive, the two classes offset by `sep`
    standard deviations along every column."""
    rng = np.random.default_rng(seed)
    n_pos = int(round(n * pos_rate))
    y = np.zeros(n, dtype=int)
    y[:n_pos] = 1
    X = rng.standard_normal((n, d))
    X[y == 1] += sep / math.sqrt(d)
    order = rng.permutation(n)
    return X[order], y[order]


def standardise(X, mean=None, std=None):
    """Centre and scale.  Which rows the mean and the std come from is the point of
    section 12."""
    if mean is None:
        mean, std = X.mean(axis=0), X.std(axis=0)
    std = np.where(std == 0, 1.0, std)
    return (X - mean) / std, mean, std


# ====================================================== 4. a model to evaluate
class LogisticRegression:
    """Logistic regression by gradient descent, with an L2 penalty.

    Same three lines as day 46's output layer: p = sigmoid(Xw + b), the gradient of
    the cross-entropy is X.T @ (p - y) / n, and the L2 term adds 2 * lam * w.
    """

    def __init__(self, l2=0.0, lr=0.5, n_iter=400):
        self.l2, self.lr, self.n_iter = l2, lr, n_iter

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n, d = X.shape
        self.w = np.zeros(d)
        self.b = 0.0
        for _ in range(self.n_iter):
            p = self.predict_proba(X)
            g = X.T @ (p - y) / n + 2.0 * self.l2 * self.w
            gb = float(np.mean(p - y))
            self.w -= self.lr * g
            self.b -= self.lr * gb
        return self

    def predict_proba(self, X):
        z = np.asarray(X, dtype=float) @ self.w + self.b
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


# ====================================================== 3. splitting the data
def kfold_indices(n, k, seed=0, shuffle=True):
    """Plain k-fold: shuffle the row numbers, cut into k near-equal blocks.

    Nothing here looks at the labels, which is the whole problem when one class is
    rare - see section 10.
    """
    idx = np.arange(n)
    if shuffle:
        np.random.default_rng(seed).shuffle(idx)
    return [np.sort(part) for part in np.array_split(idx, k)]


def stratified_kfold_indices(y, k, seed=0):
    """Same cut, done once per class, so every fold keeps the overall positive rate
    to within one row."""
    y = np.asarray(y).astype(int)
    rng = np.random.default_rng(seed)
    folds = [[] for _ in range(k)]
    for cls in (0, 1):
        members = np.where(y == cls)[0]
        rng.shuffle(members)
        for f, part in enumerate(np.array_split(members, k)):
            folds[f].extend(part.tolist())
    return [np.sort(np.array(f, dtype=int)) for f in folds]


# ====================================================== 6. cross-validation
def cv_predictions(X, y, folds, make_model, scale_inside=True):
    """Train on every fold but one, predict the held-out fold, keep the scores.

    `scale_inside=False` is the mistake in section 12: the column means and standard
    deviations are then computed once over all the rows, including the ones the model
    is about to be tested on.
    """
    X, y = np.asarray(X, dtype=float), np.asarray(y).astype(int)
    oof = np.zeros(len(y))                            # out-of-fold scores
    if not scale_inside:
        Xs, _, _ = standardise(X)
    for f in range(len(folds)):
        test = folds[f]
        train = np.setdiff1d(np.arange(len(y)), test)
        if scale_inside:
            Xtr, mu, sd = standardise(X[train])
            Xte, _, _ = standardise(X[test], mu, sd)
        else:
            Xtr, Xte = Xs[train], Xs[test]
        model = make_model().fit(Xtr, y[train])
        oof[test] = model.predict_proba(Xte)
    return oof


# ====================================================== 2. the two curves
def roc_curve(y_true, scores):
    """False-positive rate against true-positive rate, over every threshold.

    Both axes are divided by a class total, so the curve does not know that one
    class is a hundred times bigger than the other.
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)
    order = np.argsort(-s, kind='mergesort')
    y, s = y[order], s[order]
    tp = np.cumsum(y)
    fp = np.cumsum(1 - y)
    # keep only the last index of each run of equal scores: one point per threshold
    keep = np.r_[np.diff(s) != 0, True]
    P, N = int(y.sum()), int((1 - y).sum())
    tpr = np.r_[0.0, tp[keep] / P]
    fpr = np.r_[0.0, fp[keep] / N]
    return fpr, tpr


def pr_curve(y_true, scores):
    """Recall against precision.  Precision has the number of false positives in
    its denominator, so it feels every false positive, however many negatives exist.
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)
    order = np.argsort(-s, kind='mergesort')
    y, s = y[order], s[order]
    tp = np.cumsum(y)
    fp = np.cumsum(1 - y)
    keep = np.r_[np.diff(s) != 0, True]
    P = int(y.sum())
    rec = tp[keep] / P
    prec = tp[keep] / (tp[keep] + fp[keep])
    return rec, prec


def average_precision(y_true, scores):
    """The step-wise area under the precision-recall curve: sum over the positives
    of (how much recall this one added) x (precision once it is included).

    The trapezoid rule is wrong here - it interpolates between two operating points
    that no threshold produces - so this is the number everyone actually reports.
    """
    rec, prec = pr_curve(y_true, scores)
    rec = np.r_[0.0, rec]
    return float(np.sum(np.diff(rec) * prec))


def roc_auc_pairs(y_true, scores):
    """The definition, written out: over every (positive, negative) pair, how often
    does the positive score higher?  A tie counts half.  This is O(P x N) pairs."""
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float('nan')
    diff = pos[:, None] - neg[None, :]
    wins = np.sum(diff > 0) + 0.5 * np.sum(diff == 0)
    return float(wins / (len(pos) * len(neg)))


def roc_auc_rank(y_true, scores):
    """The same number from the ranks (Mann-Whitney U), in O(n log n): sort once,
    average the ranks inside each group of equal scores, and add up the positives."""
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)
    order = np.argsort(s, kind='mergesort')
    s_sorted = s[order]
    ranks = np.empty(len(s), dtype=float)
    i = 0
    while i < len(s):
        j = i
        while j + 1 < len(s) and s_sorted[j + 1] == s_sorted[i]:
            j += 1
        ranks[i:j + 1] = (i + j) / 2.0 + 1.0        # 1-based, ties share the mean rank
        i = j + 1
    r = np.empty(len(s), dtype=float)
    r[order] = ranks
    P, N = int(y.sum()), int((1 - y).sum())
    if P == 0 or N == 0:
        return float('nan')
    return float((r[y == 1].sum() - P * (P + 1) / 2) / (P * N))


# ====================================================== 8. LeetCode 169
class SolutionMajorityElement:
    """LeetCode 169 - Majority Element.

    Boyer-Moore: keep one candidate and a count; a value that occupies more than half
    the array survives every cancellation.  This is the baseline an imbalanced-data
    accuracy score is really measuring - always answer the majority class.
    """

    def majorityElement(self, nums: list[int]) -> int:
        count, candidate = 0, None
        for v in nums:
            if count == 0:
                candidate = v
            count += 1 if v == candidate else -1
        return candidate


def majority_baseline(y):
    """Accuracy of always answering the majority class - the number every other
    accuracy on this page has to beat before it means anything."""
    y = np.asarray(y).astype(int)
    maj = int(SolutionMajorityElement().majorityElement(y.tolist()))
    return maj, float(np.mean(y == maj))


X, y = imbalanced(2000, 0.01, sep=1.6, d=6, seed=0)
folds = stratified_kfold_indices(y, 5)
oof = cv_predictions(X, y, folds, lambda: LogisticRegression())
pred = predict_at(oof, 0.5)
maj, base = majority_baseline(y)
print(f'always answer {maj}        accuracy {base:.4f}')
print(f'the model at cut 0.5     accuracy {accuracy(y, pred):.4f}   '
      f'recall {recall(y, pred):.4f}')
print(f'it flagged {int(pred.sum())} row(s) and got {confusion(y, pred)[3]} right')
print(f'highest score it gave any row: {oof.max():.4f}')
print()
print(f'ROC-AUC {roc_auc_rank(y, oof):.4f}   average precision {average_precision(y, oof):.4f}')

always answer 0        accuracy 0.9900
the model at cut 0.5     accuracy 0.9895   recall 0.0000
it flagged 1 row(s) and got 0 right
highest score it gave any row: 0.6481

ROC-AUC 0.9027   average precision 0.2033


## The threshold is a number you chose, even if you never typed it

Nothing is retrained between these rows. Accuracy is highest where the model catches
nothing, and falls as the model starts being useful, because being useful means spending
true negatives to buy true positives.

In [4]:
print(f'{"cut":>6} {"flagged":>9} {"accuracy":>10} {"precision":>11} {"recall":>9} {"F1":>8}')
for t in (0.50, 0.30, 0.20, 0.15, 0.10, 0.05, 0.03, 0.02, 0.01):
    pt = predict_at(oof, t)
    print(f'{t:>6.2f} {int(pt.sum()):>9} {accuracy(y, pt):>10.4f} '
          f'{precision(y, pt):>11.4f} {recall(y, pt):>9.4f} {f1(y, pt):>8.4f}')

   cut   flagged   accuracy   precision    recall       F1
  0.50         1     0.9895      0.0000    0.0000   0.0000
  0.30         1     0.9895      0.0000    0.0000   0.0000
  0.20         3     0.9895      0.3333    0.0500   0.0870
  0.15         4     0.9890      0.2500    0.0500   0.0833
  0.10        20     0.9870      0.3500    0.3500   0.3500
  0.05        88     0.9560      0.1136    0.5000   0.1852
  0.03       220     0.8940      0.0636    0.7000   0.1167
  0.02       413     0.7995      0.0387    0.8000   0.0739
  0.01       845     0.5865      0.0225    0.9500   0.0439


## The same F1, half the cases caught

F1 is one number carrying two independent quantities, so two systems that behave nothing
alike can land on the same value — and the one that misses more than half the positives
can have the higher accuracy.

In [5]:
def equal_f1_pair(n=2000, n_pos=100):
    """Two sets of predictions with the same F1 and opposite behaviour.

    A flags 300 rows and catches 90 of the 100 positives; B flags 100 and catches 45.
    Both score F1 0.45, and B - the one that misses more than half the positives -
    has the higher accuracy.
    """
    y = np.zeros(n, dtype=int)
    y[:n_pos] = 1
    a = np.zeros(n, dtype=int)
    a[:90] = 1                       # 90 true positives
    a[n_pos:n_pos + 210] = 1         # 210 false positives -> 300 flagged
    b = np.zeros(n, dtype=int)
    b[:45] = 1                       # 45 true positives
    b[n_pos:n_pos + 55] = 1          # 55 false positives -> 100 flagged
    return y, a, b


ye, pa, pb = equal_f1_pair()
print(f'{"":<12}{"flagged":>8}{"caught":>8}{"precision":>11}{"recall":>9}{"F1":>8}{"accuracy":>11}')
for nm, pp in (('wide net', pa), ('narrow net', pb)):
    print(f'{nm:<12}{int(pp.sum()):>8}{confusion(ye, pp)[3]:>8}'
          f'{precision(ye, pp):>11.4f}{recall(ye, pp):>9.4f}{f1(ye, pp):>8.4f}'
          f'{accuracy(ye, pp):>11.4f}')

             flagged  caught  precision   recall      F1   accuracy
wide net         300      90     0.3000   0.9000  0.4500     0.8900
narrow net       100      45     0.4500   0.4500  0.4500     0.9450


## ROC-AUC and average precision rank two models in opposite orders

`broad` separates the whole population well but puts 135 negatives above its weakest
positive. `sharp` ranks half the positives above every negative and buries the
other half low - those ten outrank between 17% and 26% of the negatives, 22% on
average, which is where its ROC-AUC of 0.6082 comes from:
(10 x 1.0000 + 10 x 0.2163) / 20. The last columns are the ones a team with a review
budget actually lives on.

In [6]:
def broad_and_sharp(n=2000, n_pos=20, seed=1):
    """Two score columns over the same labels, built to disagree.

    `broad` separates the whole population well but puts 135 negatives above its
    weakest positive - section 7 measures that.  `sharp` ranks half the positives
    above every negative and buries the other half low: those ten outrank only 22% of
    the negatives, which section 7 measures.
    """
    rng = np.random.default_rng(seed)
    y = np.zeros(n, dtype=int)
    y[:n_pos] = 1
    broad = np.where(y == 1, rng.normal(0.72, 0.10, n), rng.normal(0.30, 0.20, n))
    sharp = np.where(y == 0, rng.uniform(0.00, 0.90, n), 0.0)
    half = n_pos // 2
    sharp[:half] = rng.uniform(0.95, 1.00, half)             # certain, and correct
    sharp[half:n_pos] = rng.uniform(0.15, 0.25, n_pos - half)  # buried near the bottom
    perm = rng.permutation(n)
    return y[perm], np.clip(broad[perm], 0, 1), sharp[perm]


def precision_at_k(y_true, scores, k):
    """Of the k rows with the highest scores, how many are positive?  This is the
    number a team with a fixed review budget actually lives on."""
    y = np.asarray(y_true).astype(int)
    idx = np.argsort(-np.asarray(scores, dtype=float), kind='mergesort')[:k]
    return int(y[idx].sum()), float(y[idx].mean())


yb, broad, sharp = broad_and_sharp()
print(f'{"":<8}{"ROC-AUC":>10}{"avg prec":>11}{"top 10":>9}{"top 20":>9}'
      f'{"top 50":>9}{"top 100":>10}')
for nm, sc in (('broad', broad), ('sharp', sharp)):
    hits = [precision_at_k(yb, sc, k)[0] for k in (10, 20, 50, 100)]
    print(f'{nm:<8}{roc_auc_rank(yb, sc):>10.4f}{average_precision(yb, sc):>11.4f}'
          f'{hits[0]:>9}{hits[1]:>9}{hits[2]:>9}{hits[3]:>10}')
print()
print(f'{int(yb.sum())} positives, {int((1 - yb).sum())} negatives: the ROC x-axis divides by the')
print('negatives, precision divides by however many rows you flagged')

           ROC-AUC   avg prec   top 10   top 20   top 50   top 100
broad       0.9800     0.2124        0        3       12        19
sharp       0.6082     0.5049       10       10       10        10

20 positives, 1980 negatives: the ROC x-axis divides by the
negatives, precision divides by however many rows you flagged


## Why the ROC axes are those two numbers

recall divides by rows that really are positive; FPR divides by rows that really are
negative. Neither denominator mixes the classes, so neither axis knows how many of each
class there are. Keep every positive and throw negatives away at random — the model and
its ranking never change, only the rarity of the positives does.

In [7]:
def rebalance(y_true, scores, keep_negatives, seed=0):
    """Throw away negatives at random, keep every positive.

    Nothing about the model or its ranking changes; only how rare the positives are.
    Section 8 averages this over many draws to show which metrics notice and which
    do not.
    """
    y = np.asarray(y_true).astype(int)
    rng = np.random.default_rng(seed)
    pos = np.where(y == 1)[0]
    neg = rng.permutation(np.where(y == 0)[0])[:keep_negatives]
    sel = np.r_[pos, neg]
    return y[sel], np.asarray(scores, dtype=float)[sel]


print(f'{"negatives kept":>15}{"positive rate":>15}{"ROC-AUC":>10}{"avg precision":>16}')
for keep in (1980, 495, 124, 40):
    aucs, aps, rates = [], [], []
    for r in range(50):
        yr, sr = rebalance(yb, broad, keep, seed=r)
        aucs.append(roc_auc_rank(yr, sr))
        aps.append(average_precision(yr, sr))
        rates.append(float(yr.mean()))
    print(f'{keep:>15}{np.mean(rates):>15.4f}{np.mean(aucs):>10.4f}{np.mean(aps):>16.4f}')
print()
print('each row is the mean over 50 draws; ROC-AUC barely moves, average precision does')

 negatives kept  positive rate   ROC-AUC   avg precision
           1980         0.0100    0.9800          0.2124
            495         0.0388    0.9807          0.5331
            124         0.1389    0.9797          0.8142
             40         0.3333    0.9811          0.9377

each row is the mean over 50 draws; ROC-AUC barely moves, average precision does


## AUC is a pair count

ROC-AUC is exactly *pick one positive and one negative at random — how often does the
positive score higher?*, with a tie counting half. Written that way it is a count over
pairs, and the rank formula gets the same number without building any of them. That is
the same move LeetCode 493 asks for.

In [8]:
P, N = int(yb.sum()), int((1 - yb).sum())
print(f'{P} positives x {N} negatives = {P * N:,} pairs')
print(f'every pair:  {roc_auc_pairs(yb, broad):.10f}')
print(f'from ranks:  {roc_auc_rank(yb, broad):.10f}')
print(f'difference:  {abs(roc_auc_pairs(yb, broad) - roc_auc_rank(yb, broad)):.2e}')
print()
const = np.full(10, 0.3)
lab = np.r_[np.ones(5), np.zeros(5)].astype(int)
print(f'every score identical -> every pair is a tie -> AUC {roc_auc_pairs(lab, const):.4f}')

20 positives x 1980 negatives = 39,600 pairs
every pair:  0.9800000000
from ranks:  0.9800000000
difference:  0.00e+00

every score identical -> every pair is a tie -> AUC 0.5000


## Plain k-fold hands you a fold with no positives in it

Shuffling row numbers does not look at the labels, so with 20 positives spread over 10
folds some folds get none. Recall on such a fold is 0/0.

In [9]:
plain10 = kfold_indices(len(y), 10, seed=0)
strat10 = stratified_kfold_indices(y, 10)
print('shuffle then cut  ', [int(y[f].sum()) for f in plain10])
print('cut each class    ', [int(y[f].sum()) for f in strat10])
runs = sum(1 for s in range(200)
           if min(int(y[f].sum()) for f in kfold_indices(len(y), 10, seed=s)) == 0)
print(f'\n{runs} of 200 shuffles produce at least one fold with zero positives')
dead = [f for f in plain10 if int(y[f].sum()) == 0][0]
print(f'on such a fold recall is {recall(y[dead], predict_at(oof[dead], 0.10))}')

shuffle then cut   [0, 6, 2, 1, 0, 3, 1, 4, 1, 2]
cut each class     [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

166 of 200 shuffles produce at least one fold with zero positives
on such a fold recall is nan


## Averaging the folds is not pooling the predictions

Precision, recall and F1 are ratios, and the mean of ratios is not the ratio of the
totals. The five folds flagged [2, 6, 3, 7, 2] rows; the two that flagged the fewest got them
all right and scored a precision of 1.0000, and the mean treats a fold that flagged 2
rows as equal to one that flagged 7.

In [10]:
def per_fold_metric(y, oof, folds, metric_fn, threshold=0.5):
    """One number per fold.  Folds with no positives make recall 0/0, which is where
    the nans in section 10 come from."""
    out = []
    for f in folds:
        out.append(metric_fn(y[f], predict_at(oof[f], threshold)))
    return np.array(out, dtype=float)


pf = per_fold_metric(y, oof, folds, precision, 0.10)
print('per-fold precision', np.round(pf, 4).tolist())
print(f'mean of the folds  {np.nanmean(pf):.4f}')
print(f'pooled             {precision(y, predict_at(oof, 0.10)):.4f}')

per-fold precision [1.0, 0.1667, 0.3333, 0.1429, 1.0]
mean of the folds  0.5286
pooled             0.3500


## Three kinds of leakage, measured and ranked

They are not equally bad, and saying so is the point: two of them turn noise into a
result, and one moves the score by almost nothing.

Note what changes between (b) and (c): (b) is 60 rows of pure noise scored by a nearest
centroid, and its two numbers are means over 20 independently drawn tables. (c) is a
different 60 rows — 8 columns with real signal — scored by logistic regression, and the
only thing that differs between its two numbers is where the column means and standard
deviations were fitted.

In [11]:
def grouped_rows(n_groups=150, per=4, d=6, within=0.25, seed=11):
    """Four rows per customer.  The label belongs to the customer, and the four rows
    sit close together, so a random split puts near-copies on both sides of it."""
    rng = np.random.default_rng(seed)
    g = np.repeat(np.arange(n_groups), per)
    glabel = rng.integers(0, 2, n_groups)
    gshift = rng.standard_normal((n_groups, d)) * 2.0
    X = gshift[g] + rng.standard_normal((n_groups * per, d)) * within
    return X, glabel[g], g


def group_kfold_indices(groups, k):
    """Keep every row of a group together.  Rows from one customer, one patient or
    one document split across train and test are the same row twice."""
    uniq = np.unique(groups)
    buckets = [[] for _ in range(k)]
    sizes = [0] * k
    for g in uniq:                                   # largest group to the smallest bucket
        rows = np.where(groups == g)[0]
        t = int(np.argmin(sizes))
        buckets[t].extend(rows.tolist())
        sizes[t] += len(rows)
    return [np.sort(np.array(b, dtype=int)) for b in buckets]


def one_nearest_neighbour(Xtr, ytr, Xte):
    """Copy the label of the closest training row.  It has no parameters, so when it
    scores well the only thing it can have found is a row that looks like one it
    already saw - which is exactly what group leakage hands it."""
    d2 = ((Xte[:, None, :] - Xtr[None, :, :]) ** 2).sum(-1)
    return ytr[np.argmin(d2, axis=1)]


def pure_noise_table(n=60, d=400, seed=3):
    """n rows, d columns of noise, a coin-flip label.  There is nothing to learn, so
    any score above chance in section 12 was manufactured by the procedure."""
    rng = np.random.default_rng(seed)
    return rng.standard_normal((n, d)), rng.integers(0, 2, n)


def select_by_correlation(X, y, k):
    """Keep the k columns most correlated with the label.  Run on all the rows this
    is the fastest way to invent a signal that is not there (section 12)."""
    Xc = X - X.mean(axis=0)
    yc = y - y.mean()
    denom = np.sqrt((Xc ** 2).sum(axis=0) * (yc ** 2).sum()) + 1e-12
    corr = np.abs(Xc.T @ yc / denom)
    return np.argsort(-corr)[:k]


def nearest_centroid_accuracy(Xtr, ytr, Xte, yte):
    """A classifier with no knobs at all, so that section 12's numbers cannot be
    blamed on the model: put each row with the class centre it sits closest to."""
    c0 = Xtr[ytr == 0].mean(axis=0)
    c1 = Xtr[ytr == 1].mean(axis=0)
    d0 = ((Xte - c0) ** 2).sum(axis=1)
    d1 = ((Xte - c1) ** 2).sum(axis=1)
    return float(np.mean((d1 < d0).astype(int) == yte))


Xg, yg, g = grouped_rows()
ng = len(yg)
def knn_cv(fl):
    return float(np.mean([np.mean(one_nearest_neighbour(
        Xg[np.setdiff1d(np.arange(ng), f)], yg[np.setdiff1d(np.arange(ng), f)],
        Xg[f]) == yg[f]) for f in fl]))
print(f'(a) same customer both sides   random folds {knn_cv(kfold_indices(ng, 5, seed=0)):.4f}'
      f'   group folds {knn_cv(group_kfold_indices(g, 5)):.4f}')

out_s, in_s = [], []
for seed in range(20):
    Xn, yn = pure_noise_table(60, 400, seed=seed)
    fl = stratified_kfold_indices(yn, 5)
    cols = select_by_correlation(Xn, yn, 5)
    a_, b_ = [], []
    for f in fl:
        tr = np.setdiff1d(np.arange(60), f)
        a_.append(nearest_centroid_accuracy(Xn[tr][:, cols], yn[tr], Xn[f][:, cols], yn[f]))
        c2 = select_by_correlation(Xn[tr], yn[tr], 5)
        b_.append(nearest_centroid_accuracy(Xn[tr][:, c2], yn[tr], Xn[f][:, c2], yn[f]))
    out_s.append(np.mean(a_)); in_s.append(np.mean(b_))
print(f'(b) columns picked first       all rows     {np.mean(out_s):.4f}'
      f'   inside folds {np.mean(in_s):.4f}   (true 0.5)')

Xs, ys = imbalanced(60, 0.3, sep=1.2, d=8, seed=5)
fl = stratified_kfold_indices(ys, 5)
ins = cv_predictions(Xs, ys, fl, lambda: LogisticRegression(), scale_inside=True)
outs = cv_predictions(Xs, ys, fl, lambda: LogisticRegression(), scale_inside=False)
print(f'(c) scaled over every row      leaking     {accuracy(ys, predict_at(outs, .5)):.4f}'
      f'    honest      {accuracy(ys, predict_at(ins, .5)):.4f}'
      f'   max score gap {np.abs(ins - outs).max():.2e}')

(a) same customer both sides   random folds 0.9900   group folds 0.4750
(b) columns picked first       all rows     0.7837   inside folds 0.5169   (true 0.5)
(c) scaled over every row      leaking     0.7167    honest      0.7167   max score gap 1.50e-04


## The set you chose on is no longer a test set

Every model below is a coin flip and so is every label, so the true accuracy of all of
them is 0.5. The first column rises anyway, because the maximum of m noisy measurements
grows with m.

In [12]:
def best_of_m_on_one_set(m, n=200, reps=40, seed=0):
    """Score m coin-flip models on one labelled set, keep the best, then score that
    same model on a second set drawn the same way.

    Nothing here has any signal: both label sets are coin flips and so is every
    model.  Whatever the first number rises to is the amount of optimism you buy by
    choosing on a set and then reporting the score from that same set.
    """
    picked, fresh = [], []
    for rep in range(reps):
        r = np.random.default_rng(seed + rep)
        y_pick, y_fresh = r.integers(0, 2, n), r.integers(0, 2, n)
        best = (-1.0, 0.0)
        for _ in range(m):
            pred = r.integers(0, 2, n)
            cand = (float(np.mean(pred == y_pick)), float(np.mean(pred == y_fresh)))
            if cand > best:
                best = cand
        picked.append(best[0])
        fresh.append(best[1])
    return float(np.mean(picked)), float(np.mean(fresh))


print(f'{"models tried":>13}{"best, on the set you chose on":>32}{"same model, fresh set":>24}')
for m in (1, 10, 50, 200):
    a_, b_ = best_of_m_on_one_set(m, n=200, reps=40, seed=1000)
    print(f'{m:>13}{a_:>32.4f}{b_:>24.4f}')

 models tried   best, on the set you chose on   same model, fresh set
            1                          0.5061                  0.4940
           10                          0.5563                  0.5044
           50                          0.5800                  0.5064
          200                          0.5981                  0.4952


## LeetCode 493 — Reverse Pairs

Count the pairs `i < j` with `nums[i] > 2 * nums[j]`. The answer is a count over pairs,
and the one thing you must not do is build them. Merge sort settles a whole block per
comparison — but the counting has to happen in its own pass, because the merge advances
on `left[i] <= right[j]` while the pairs need `left[i] > 2 * right[j]`.

In [13]:
# ====================================================== 7. LeetCode 493
class SolutionReversePairs:
    """LeetCode 493 - Reverse Pairs.

    Count the pairs i < j with nums[i] > 2 * nums[j].  Counting the pairs where a
    positive outranks a negative - which is what AUC is - has the same shape, and
    the same O(n^2) trap: the answer is a count over pairs, but you must never build
    the pairs.  Merge sort counts a whole block at a time while it sorts.
    """

    def reversePairs(self, nums: list[int]) -> int:
        def sort_count(a):
            if len(a) < 2:
                return a, 0
            mid = len(a) // 2
            left, cl = sort_count(a[:mid])
            right, cr = sort_count(a[mid:])
            count = cl + cr
            j = 0
            for v in left:                       # both halves are sorted, so j only moves forward
                while j < len(right) and v > 2 * right[j]:
                    j += 1
                count += j
            merged, i, j2 = [], 0, 0
            while i < len(left) and j2 < len(right):
                if left[i] <= right[j2]:
                    merged.append(left[i]); i += 1
                else:
                    merged.append(right[j2]); j2 += 1
            merged.extend(left[i:]); merged.extend(right[j2:])
            return merged, count
        return sort_count(list(nums))[1]


def reverse_pairs_during_merge(nums):
    """The tempting shortcut: count while merging, instead of in a pass of its own.

    The merge advances on `left[i] <= right[j]`, but the pairs we want are
    `left[i] > 2 * right[j]`.  Those are two different orders, so the pointer stops
    in the wrong place and the count comes out low.  It still returns a number.
    """
    def sort_count(a):
        if len(a) < 2:
            return a, 0
        mid = len(a) // 2
        left, cl = sort_count(a[:mid])
        right, cr = sort_count(a[mid:])
        count, merged, i, j = cl + cr, [], 0, 0
        while i < len(left) and j < len(right):
            if left[i] <= right[j]:
                merged.append(left[i]); i += 1
            else:
                if left[i] > 2 * right[j]:
                    count += len(left) - i
                merged.append(right[j]); j += 1
        merged.extend(left[i:]); merged.extend(right[j:])
        return merged, count
    return sort_count(list(nums))[1]


def reverse_pairs_brute(nums):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i + 1, n) if nums[i] > 2 * nums[j])


for a in ([1, 3, 2, 3, 1], [2, 4, 3, 5, 1]):
    print(f'{a} -> {SolutionReversePairs().reversePairs(a)}')

bad, wrong = None, 0
for seed in range(2000):
    r = np.random.default_rng(seed)
    cand = r.integers(-20, 20, int(r.integers(2, 10))).tolist()
    got, want = reverse_pairs_during_merge(cand), reverse_pairs_brute(cand)
    if got != want:
        wrong += 1
        bad = bad or (cand, got, want)
print(f'\ncounting inside the merge instead: wrong on {wrong} of 2,000 random arrays')
print(f'first one {bad[0]} -> says {bad[1]}, answer {bad[2]}')

[1, 3, 2, 3, 1] -> 2
[2, 4, 3, 5, 1] -> 3

counting inside the merge instead: wrong on 1051 of 2,000 random arrays
first one [5, 0, -10, -8, -19, -17, -20, -13] -> says 23, answer 28


## LeetCode 169 — Majority Element

Return the value appearing more than `n / 2` times. Boyer-Moore keeps one candidate and
one count. It is the baseline the accuracy above was really measuring — and without
LeetCode's promise that a majority exists, it still returns a number.

In [14]:
def majority_if_any(nums):
    """LeetCode 169 promises a majority exists.  Without that promise Boyer-Moore
    still returns something, so the follow-up is a second pass that checks it."""
    c = SolutionMajorityElement().majorityElement(nums)
    return c if sum(1 for v in nums if v == c) * 2 > len(nums) else None


for a in ([3, 2, 3], [2, 2, 1, 1, 1, 2, 2]):
    print(f'{a} -> {SolutionMajorityElement().majorityElement(a)}')
print(f'\non the 2,000-row table it answers {maj}, right {base:.4f} of the time')
print(f'[1, 2, 3] -> Boyer-Moore says {SolutionMajorityElement().majorityElement([1, 2, 3])}, '
      f'checked with a second pass: {majority_if_any([1, 2, 3])}')

[3, 2, 3] -> 3
[2, 2, 1, 1, 1, 2, 2] -> 2

on the 2,000-row table it answers 0, right 0.9900 of the time
[1, 2, 3] -> Boyer-Moore says 3, checked with a second pass: None


## Tests

Everything above, re-derived from scratch and asserted.

In [15]:
# ====================================================== 9. assertions
def check():
    rng = np.random.default_rng(0)

    # the four counts, by hand
    y = np.array([1, 1, 1, 0, 0, 0, 0, 0])
    p = np.array([1, 1, 0, 1, 0, 0, 0, 0])
    assert confusion(y, p) == (4, 1, 1, 2)
    assert abs(accuracy(y, p) - 6 / 8) < 1e-12
    assert abs(precision(y, p) - 2 / 3) < 1e-12
    assert abs(recall(y, p) - 2 / 3) < 1e-12
    assert abs(f1(y, p) - 2 / 3) < 1e-12
    assert abs(specificity(y, p) - 4 / 5) < 1e-12
    assert abs(balanced_accuracy(y, p) - (2 / 3 + 4 / 5) / 2) < 1e-12

    # the apple basket: each ratio divides by a different thing
    ya, pa_ = apples_and_oranges()
    assert confusion(ya, pa_) == (88, 2, 4, 6)
    assert abs(accuracy(ya, pa_) - 0.94) < 1e-12
    assert abs(precision(ya, pa_) - 6 / 8) < 1e-12        # of the apple SIDE
    assert abs(recall(ya, pa_) - 6 / 10) < 1e-12          # of ALL the apples
    assert abs(false_positive_rate(ya, pa_) - 2 / 90) < 1e-12
    # FPR and specificity are the same number seen from the two ends
    assert abs(false_positive_rate(ya, pa_) + specificity(ya, pa_) - 1.0) < 1e-12
    # the ROC curve is exactly (FPR, recall) at every threshold - check it point by point
    yy = np.array([1, 1, 0, 0, 1, 0])
    ss = np.array([.95, .40, .80, .05, .60, .30])
    fpr_c, tpr_c = roc_curve(yy, ss)
    for t in sorted(set(ss.tolist())):
        pt = predict_at(ss, t)
        pair = (false_positive_rate(yy, pt), recall(yy, pt))
        assert any(abs(pair[0] - a_) < 1e-12 and abs(pair[1] - b_) < 1e-12
                   for a_, b_ in zip(fpr_c, tpr_c)), (t, pair)

    # dropping negatives at random moves average precision and leaves ROC-AUC alone
    yb_, broad_, _ = broad_and_sharp()
    base_auc, base_ap = roc_auc_rank(yb_, broad_), average_precision(yb_, broad_)
    aucs, aps = [], []
    for r in range(50):
        yr, sr = rebalance(yb_, broad_, 124, seed=r)
        assert int(yr.sum()) == int(yb_.sum()) and len(yr) == int(yb_.sum()) + 124
        aucs.append(roc_auc_rank(yr, sr))
        aps.append(average_precision(yr, sr))
    assert abs(np.mean(aucs) - base_auc) < 0.01      # the ROC axes do not notice
    assert np.mean(aps) - base_ap > 0.5              # precision very much does

    # predicting one class everywhere: precision and MCC go undefined, accuracy does not
    allneg = np.zeros(8, dtype=int)
    assert math.isnan(precision(y, allneg)) and math.isnan(mcc(y, allneg))
    assert abs(accuracy(y, allneg) - 5 / 8) < 1e-12

    # the pair definition of AUC and the rank formula agree, ties included
    for seed in range(20):
        r = np.random.default_rng(seed)
        n = int(r.integers(20, 120))
        yy = r.integers(0, 2, n)
        if yy.sum() == 0 or yy.sum() == n:
            continue
        ss = np.round(r.normal(size=n), 1)            # rounding manufactures ties
        assert abs(roc_auc_pairs(yy, ss) - roc_auc_rank(yy, ss)) < 1e-12
    # a constant score is a coin flip by definition: every pair is a tie
    assert abs(roc_auc_pairs(np.r_[np.ones(5), np.zeros(5)].astype(int),
                             np.full(10, 0.3)) - 0.5) < 1e-12

    # average precision is bounded, and a perfect ranking scores exactly 1
    yp = np.array([1, 1, 1, 0, 0, 0, 0])
    assert abs(average_precision(yp, np.array([.9, .8, .7, .6, .5, .4, .3])) - 1.0) < 1e-12
    yr, sr = rng.integers(0, 2, 50), rng.random(50)
    assert 0.0 <= average_precision(yr, sr) <= 1.0

    # folds really are a partition
    n, k = 103, 5
    ys = np.r_[np.ones(9), np.zeros(94)].astype(int)
    plain = kfold_indices(n, k, seed=1)
    strat = stratified_kfold_indices(ys, k)
    for folds in (plain, strat):
        assert sorted(np.concatenate(folds).tolist()) == list(range(n))
    # plain k-fold makes the fold sizes as equal as they can be
    assert max(len(f) for f in plain) - min(len(f) for f in plain) <= 1
    # stratified k-fold splits each class separately, so the sizes can differ by one
    # per class rather than one overall - here 21/21/21/21/19 for two classes
    assert max(len(f) for f in strat) - min(len(f) for f in strat) <= 2
    counts = [int(ys[f].sum()) for f in strat]
    assert max(counts) - min(counts) <= 1 and sum(counts) == 9

    # groups never straddle a fold boundary
    groups = np.repeat(np.arange(12), 4)
    for f in group_kfold_indices(groups, 4):
        others = np.setdiff1d(np.arange(len(groups)), f)
        assert not set(groups[f].tolist()) & set(groups[others].tolist())

    # time-series splits only ever test on rows after the ones they trained on
    for tr, te in time_series_splits(100, 4):
        assert tr.max() < te.min()

    # LeetCode 493 against brute force
    for seed in range(300):
        r = np.random.default_rng(seed)
        a = r.integers(-20, 20, int(r.integers(0, 12))).tolist()
        assert SolutionReversePairs().reversePairs(a) == reverse_pairs_brute(a)
    assert SolutionReversePairs().reversePairs([1, 3, 2, 3, 1]) == 2
    assert SolutionReversePairs().reversePairs([2, 4, 3, 5, 1]) == 3
    # counting inside the merge never overcounts - it stops early, it does not invent pairs
    for seed in range(400):
        r = np.random.default_rng(seed)
        a = r.integers(-20, 20, int(r.integers(2, 10))).tolist()
        assert reverse_pairs_during_merge(a) <= reverse_pairs_brute(a)

    # LeetCode 169 against a count, and the follow-up that 169 does not ask for
    for seed in range(300):
        r = np.random.default_rng(seed + 1000)
        n = int(r.integers(1, 15))
        maj = int(r.integers(0, 5))
        a = [maj] * (n // 2 + 1) + r.integers(0, 5, n - n // 2 - 1).tolist()
        r.shuffle(a)
        assert SolutionMajorityElement().majorityElement(a) == maj
    assert majority_if_any([1, 2, 3]) is None
    assert majority_if_any([2, 2, 1, 1, 1, 2, 2]) == 2
    return True


def auc_trapezoid(x, y):
    """Area under a curve given as points, by the trapezoid rule."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    return float(np.trapezoid(y, x)) if hasattr(np, 'trapezoid') else float(np.trapz(y, x))


def time_series_splits(n, k):
    """Train on the past, test on the next block.  Never shuffle a time series:
    a shuffled split lets the model read rows that happen after the ones it predicts.
    """
    bounds = np.linspace(0, n, k + 2).astype(int)
    out = []
    for i in range(1, k + 1):
        out.append((np.arange(0, bounds[i]), np.arange(bounds[i], bounds[i + 1])))
    return out


def sparse_signal(n, d=40, n_signal=4, seed=7):
    """n rows, d columns, only the first few of which affect the label.  With d close
    to n a linear model can fit the training rows exactly using the columns that mean
    nothing, which is what the L2 penalty in section 13 is there to stop."""
    rng = np.random.default_rng(seed)
    w = np.zeros(d)
    w[:n_signal] = [2.0, -2.0, 1.5, -1.5][:n_signal]
    X = rng.standard_normal((n, d))
    p = 1.0 / (1.0 + np.exp(-(X @ w)))
    return X, (rng.random(n) < p).astype(int)


print('check() ->', check())

check() -> True
